In [1]:
# Install project in editable mode
!pip install -e ".[dev]"


Obtaining file:///content/drive/Othercomputers/My%20Laptop%20%281%29/small-code-models/notebooks
ERROR: file:///content/drive/Othercomputers/My%20Laptop%20%281%29/small-code-models/notebooks does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


## Shared library overview
This repository exposes a reusable package (`small_code_models`) with:
- `data.py`: dataset loading and `CloneDetectionDataset`
- `metrics.py`: consistent clone-detection metrics
- `trainer.py`: a thin Hugging Face trainer wrapper for reproducible runs


In [2]:
import numpy as np
from transformers import AutoTokenizer
from small_code_models.data import CloneDetectionDataset
from small_code_models.metrics import compute_metrics

# Tiny synthetic dataset: 5 clone pairs + 5 non-clone pairs
pairs = [
    ("def add(a,b): return a+b", "def add(x,y): return x+y"),
    ("for i in range(10): print(i)", "for j in range(10): print(j)"),
    ("if x>0: x-=1", "if n>0: n-=1"),
    ("while n: n-=1", "while k: k-=1"),
    ("return sorted(nums)", "return sorted(values)"),
    ("def add(a,b): return a+b", "def multiply(a,b): return a*b"),
    ("print('hello')", "total = sum(items)"),
    ("if flag: run()", "class A: pass"),
    ("x = [i*i for i in xs]", "raise ValueError('bad')"),
    ("return min(nums)", "with open(p) as f: data=f.read()"),
]
labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

tokenizer = AutoTokenizer.from_pretrained('microsoft/codebert-base')
dataset = CloneDetectionDataset(tokenizer=tokenizer, pairs=pairs, labels=labels, max_length=128)
print('Dataset size:', len(dataset))

# Metric demo with toy logits
toy_logits = np.array([[0.2, 0.8]] * 5 + [[0.8, 0.2]] * 5)
metrics = compute_metrics((toy_logits, np.array(labels)))
metrics


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Dataset size: 10


{'accuracy': 1.0,
 'balanced_accuracy': 1.0,
 'f1': 1.0,
 'precision': 1.0,
 'recall': 1.0,
 'mcc': 1.0,
 'roc_auc': 1.0,
 'pr_auc': 1.0,
 'specificity': 1.0,
 'negative_predictive_value': 1.0,
 'false_positive_rate': 0.0,
 'false_negative_rate': 0.0,
 'brier_score': 0.12555945331754725,
 'log_loss': 0.4374879504858857,
 'expected_calibration_error': 0.35434369377420455,
 'support': 10.0,
 'positive_support': 5.0,
 'negative_support': 5.0,
 'true_positive': 5.0,
 'true_negative': 5.0,
 'false_positive': 0.0,
 'false_negative': 0.0}

In [3]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('microsoft/codebert-base', num_labels=2)
batch = tokenizer(pairs[0][0], pairs[0][1], return_tensors='pt', truncation=True, max_length=128)
outputs = model(**batch)
print('Logits shape:', outputs.logits.shape)
outputs.logits


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: torch.Size([1, 2])


tensor([[0.0270, 0.1115]], grad_fn=<AddmmBackward0>)

## Switching to full benchmarks
Use any benchmark script with dataset and output directories:
```bash
python bcb_detection_models/codebert-bcb-01.py --data_dir /path/to/bcb --output_dir results/codebert_bcb
```
For complete reproduction across all datasets/models:
```bash
bash scripts/run_all_benchmarks.sh /path/to/datasets
```
